In [ ]:
import threading
import time
import numpy as np
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import pyzed.sl as sl
import motors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
robot  = motors.MotorsYukon(mecanum=False)


In [ ]:
IMAGE_WIDTH       = 672.0
MODEL_SAVE_PATH   = 'regression_model_simple.pth'
FORWARD_THRESHOLD = 40
MIN_SPEED         = 0.8
MAX_SPEED         = 1
CENTER_X          = 420
KP                = 0.5
K_ANGLE           = 30.0   # pixels of steering correction per unit line slope
MIN_YELLOW_PIXELS = 150

# HSV bounds for the yellow track line (OpenCV H range: 0–179)
YELLOW_HSV_LOWER = np.array([18, 100, 100])
YELLOW_HSV_UPPER = np.array([35, 255, 255])

def line_is_visible(frame_bgra: np.ndarray) -> bool:
    hsv  = cv2.cvtColor(frame_bgra[:, :, :3], cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, YELLOW_HSV_LOWER, YELLOW_HSV_UPPER)
    return int(np.count_nonzero(mask)) >= MIN_YELLOW_PIXELS

def compute_turn_speed(total_steering_error: float) -> float:
    raw = abs(total_steering_error) / (IMAGE_WIDTH / 2)
    return float(max(MIN_SPEED, min(MAX_SPEED, raw)))

# Rebuild the same architecture used during training
model = models.mobilenet_v3_small(weights=None)
num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, 1)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=True))
model = model.to(device)
model.eval()

# Same as the val_transform used during training (no augmentation)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f'Model loaded from {MODEL_SAVE_PATH}')


In [ ]:
class Camera():
    def __init__(self):
        self.zed = sl.Camera()

        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA

        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera Open: ' + repr(status) + '. Exit program.')
            self.zed.close()
            exit(1)

        self.runtime               = sl.RuntimeParameters()
        self.thread_running_flag   = False
        self.latest_frame          = None
        self.latest_track_center_x = None
        self.line_lost             = False
        self.last_turn             = None  # 'left' or 'right'
        self.proposed_turn         = None
        self.turn_change_start_time = 0.0

        camera_info = self.zed.get_camera_information()
        self.width  = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_running_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                frame_bgra = self.image.get_data()

                frame_bgra[:188, :] = 0
                frame_bgra[:, :248] = 0
                frame_bgra[:, 604:] = 0

                self.latest_frame = frame_bgra.copy()

                if not line_is_visible(frame_bgra):
                    self.line_lost = True
                    if self.last_turn == 'left':
                        robot.left(MIN_SPEED)
                    elif self.last_turn == 'right':
                        robot.right(MIN_SPEED)
                    continue

                self.line_lost = False

                frame_rgb    = cv2.cvtColor(frame_bgra[:, :, :3], cv2.COLOR_BGR2RGB)
                pil_image    = Image.fromarray(frame_rgb)
                input_tensor = preprocess(pil_image).unsqueeze(0).to(device)

                with torch.no_grad():
                    pred_norm = model(input_tensor).item()

                track_center_x = pred_norm * IMAGE_WIDTH
                self.latest_track_center_x = track_center_x

                error = track_center_x - CENTER_X

                # Slope correction: fit a line through the yellow mask pixels 
                hsv_frame   = cv2.cvtColor(frame_bgra[:, :, :3], cv2.COLOR_BGR2HSV)
                slope_mask  = cv2.inRange(hsv_frame, YELLOW_HSV_LOWER, YELLOW_HSV_UPPER)
                mask_pixels = np.argwhere(slope_mask == 255)  # [:, 0]=row, [:, 1]=col
                if len(mask_pixels) >= 20:
                    rows = mask_pixels[:, 0].astype(np.float64)
                    cols = mask_pixels[:, 1].astype(np.float64)
                    slope, _ = np.polyfit(rows, cols, 1)
                    error += -slope * K_ANGLE

                total_steering_error = KP * error
                speed = compute_turn_speed(total_steering_error)

                # Determine what direction the current frame is suggesting
                current_detected_turn = 'left' if total_steering_error < 0 else 'right'

                # Initialize the states on the very first frame
                if self.last_turn is None:
                    self.last_turn     = current_detected_turn
                    self.proposed_turn = current_detected_turn

                # Buffer Logic
                if current_detected_turn != self.last_turn:
                    # The direction has changed from our official state
                    if self.proposed_turn != current_detected_turn:
                        # just noticed this new direction, start the clock
                        self.proposed_turn          = current_detected_turn
                        self.turn_change_start_time = time.time()
                    elif (time.time() - self.turn_change_start_time) >= 0.2: # commit change if persisted for mor than 0.2 seconds
                        self.last_turn = current_detected_turn
                else:
                    self.proposed_turn = current_detected_turn

                if abs(total_steering_error) < FORWARD_THRESHOLD:
                    robot.forward(speed)
                elif total_steering_error < 0:
                    robot.left(speed)
                else:
                    robot.right(speed)

    def start(self):
        if not self.thread_running_flag:
            self.thread_running_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_running_flag:
            self.thread_running_flag = False
            self.thread.join()
            robot.stop()

    def close(self):
        self.stop()
        self.zed.close()


camera = Camera()


In [ ]:
camera.start()

In [ ]:
camera.stop()